# Follow-up experiments: cells D and E (Kaggle T4 x2)

Runs **E1** (2×2 design, cells D and E) → **E3** → **E2** → **X1** (cell E), because the benchmark runs of cells D and E were made on Kaggle.
About 9.5 GPU hours, i.e. about **5 hours** per session on 2 GPUs.

**Settings (right panel):** Accelerator `GPU T4 x2` · Internet `On` · Persistence `Files only`.

**Input:** (1) a Kaggle dataset holding this repository (with `run_exp.py`); (2) optionally the CIFAR-10 / CIFAR-100 datasets
(otherwise the runner downloads them, Internet On); (3) from the 2nd session on: the Output of this notebook.

**Run:** first with `RUN_CHECKS = True` and an interactive Run All to see SELFTEST PASSED; then `RUN_CHECKS = False` and
**Save Version → Save & Run All (Commit)**.

In [ ]:
# === 1. Parameters ===
CELLS         = ['D', 'E']
EXPS          = ['E1', 'E3', 'E2', 'X1']   # priority order
RUN_CHECKS    = False    # True the first time
MAX_HOURS     = 10.5     # do not start new runs after this many hours
TIME_BUDGET_H = 11.25    # hard stop if still running (Kaggle ends sessions at ~12 hours)
N_WORKERS     = 2

WORK    = '/kaggle/working'
CODE    = f'{WORK}/adam-bench'
DATA    = f'{WORK}/data'
RESULTS = f'{WORK}/results_exp'

In [ ]:
# === 2. Copy code and data (if present in Input), restore results of the previous session ===
import glob, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

runner = [h for h in glob.glob('/kaggle/input/**/run_exp.py', recursive=True) if '/results' not in h]
if not runner:
    raise SystemExit('run_exp.py NOT found in Input; attach the code dataset.')
if os.path.exists(CODE):
    shutil.rmtree(CODE)
shutil.copytree(os.path.dirname(runner[0]), CODE,
                ignore=shutil.ignore_patterns('results*', 'data', '__pycache__'))
print('Code:', os.path.dirname(runner[0]))

os.makedirs(DATA, exist_ok=True)
for folder, tgz in (('cifar-10-batches-py', 'cifar-10-python.tar.gz'), ('cifar-100-python', 'cifar-100-python.tar.gz')):
    dst = os.path.join(DATA, folder)
    if os.path.exists(dst):
        continue
    dirs = [d for d in glob.glob(f'/kaggle/input/**/{folder}', recursive=True) if os.path.isdir(d)]
    tars = glob.glob(f'/kaggle/input/**/{tgz}', recursive=True)
    if dirs:
        shutil.copytree(dirs[0], dst); print('Data: copied', dirs[0])
    elif tars:
        shutil.copy(tars[0], DATA)
        with tarfile.open(tars[0]) as t: t.extractall(DATA)
        print('Data: unpacked', tars[0])
    else:
        print(f'Data: {folder} not in Input; the runner will download it (needs Internet On)')

def count_done(d):
    return len(glob.glob(os.path.join(d, '**', 'result.json'), recursive=True))
if os.path.isdir(RESULTS):
    print('Using existing results_exp:', count_done(RESULTS), 'runs')
else:
    cands = [c for c in glob.glob('/kaggle/input/**/results_exp', recursive=True) if os.path.isdir(c)]
    if cands:
        best = max(cands, key=count_done)
        shutil.copytree(best, RESULTS)
        print('Restored from', best, ':', count_done(RESULTS), 'runs')
    else:
        print('Starting from scratch')

In [ ]:
# === Pin the torch + cuDNN versions of the benchmark (the runner is a subprocess, so NO restart is needed) ===
import subprocess, sys
WANT = {'B': ('2.11.0', '0.26.0', 'cu130', 92700),
        'D': ('2.11.0', '0.26.0', 'cu128', 91900),
        'E': ('2.11.0', '0.26.0', 'cu128', 91900)}
tv, vv, cu, cudnn_want = WANT[CELLS[0]]
want = f'{tv}+{cu}'

def probe():
    out = subprocess.run([sys.executable, '-c',
                          'import torch; print(torch.__version__); print(torch.backends.cudnn.version()); '
                          'print("|".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())))'],
                         capture_output=True, text=True).stdout.splitlines()
    return out[0], int(out[1]), out[2] if len(out) > 2 else ''

have, cudnn, gpus = probe()
print('installed torch:', have, '| required:', want)
if have != want:
    print('Reinstalling torch to match the benchmark (a few minutes)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'torch=={tv}+{cu}', f'torchvision=={vv}+{cu}',
                           '--index-url', f'https://download.pytorch.org/whl/{cu}'])
    have, cudnn, gpus = probe()
    print('torch after install:', have)
assert have == want, f'Could not install torch {want}; stopping.'

if cudnn != cudnn_want:
    pkg = 'nvidia-cudnn-cu13' if cu.startswith('cu13') else 'nvidia-cudnn-cu12'
    spec = f'{pkg}=={cudnn_want // 10000}.{cudnn_want // 100 % 100}.*'
    print(f'cuDNN is {cudnn}, required {cudnn_want} -> installing {spec}')
    subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', spec])
    have, cudnn, gpus = probe()
print('torch', have, '| cuDNN', cudnn, '(required', cudnn_want, ') | GPU:', gpus)
assert 'T4' in gpus, 'Not a T4 GPU; switch the runtime/accelerator to T4.'
FORCE_ENV = cudnn != cudnn_want
if FORCE_ENV:
    print('WARNING: could not match the cuDNN version. Continuing with --force-env: torch and GPU match, '
          'only the cuDNN patch differs; results are statistically comparable but may not match digit for digit.')
else:
    print('Environment matches the benchmark.')


In [ ]:
# === 4. Multi-GPU launcher with a time budget ===
import signal, threading
os.chdir(CODE)
T0 = time.time()
n_gpu = int(subprocess.run([sys.executable, '-c', 'import torch; print(torch.cuda.device_count())'],
                           capture_output=True, text=True).stdout.strip() or 0)
N = min(N_WORKERS, n_gpu)
if N == 0:
    raise SystemExit('No GPU. Settings → Accelerator → GPU T4 x2.')
print('Using', N, 'GPU(s)')

def run_parallel(args, n, budget_s, log):
    os.makedirs(os.path.dirname(log), exist_ok=True)
    f = open(log, 'a'); lk = threading.Lock(); procs = []
    def pump(p, k):
        for line in p.stdout:
            if not line.startswith('['):
                line = f'[w{k}] ' + line
            with lk:
                print(line, end='', flush=True); f.write(line); f.flush()
    for k in range(n):
        e = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k), PYTHONUNBUFFERED='1')
        cmd = [sys.executable, 'run_exp.py'] + args + ['--worker', str(k), '--workers', str(n)]
        print(f'$ CUDA_VISIBLE_DEVICES={k}', ' '.join(cmd), flush=True)
        p = subprocess.Popen(cmd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, start_new_session=True)
        th = threading.Thread(target=pump, args=(p, k), daemon=True); th.start()
        procs.append((p, th)); time.sleep(20)
    while any(p.poll() is None for p, _ in procs):
        if time.time() - T0 > budget_s:
            print('\n>>> Time budget exhausted; stopping the workers (checkpoints are safe).', flush=True)
            for p, _ in procs:
                if p.poll() is None: os.killpg(p.pid, signal.SIGTERM)
            for p, _ in procs:
                try: p.wait(120)
                except subprocess.TimeoutExpired: os.killpg(p.pid, signal.SIGKILL)
            break
        time.sleep(10)
    rcs = [p.wait() for p, _ in procs]
    for _, th in procs: th.join(5)
    f.close()
    print(f'[exit {rcs}] after {(time.time()-T0)/3600:.2f} h', flush=True)
    return max(rcs, key=abs)

In [ ]:
# === 5. Self-tests (first time) ===
if RUN_CHECKS:
    assert subprocess.call([sys.executable, 'run_exp.py', '--selftest', '--out', '/kaggle/tmp_st']) == 0, 'run_exp SELFTEST failed'
    assert subprocess.call([sys.executable, 'run_cell.py', '--selftest', '--out', '/kaggle/tmp_st']) == 0, 'run_cell SELFTEST failed'

In [ ]:
# === 6. Run ===
subprocess.call([sys.executable, 'run_exp.py', '--plan', '--cells', *CELLS])
if not RUN_CHECKS:
    args = ['--exp', *EXPS, '--cells', *CELLS, '--out', RESULTS, '--data', DATA, '--max-hours', str(MAX_HOURS)] + (['--force-env'] if FORCE_ENV else [])
    rc = run_parallel(args, N, TIME_BUDGET_H * 3600, os.path.join(RESULTS, 'kaggle_log.txt'))
    if rc == 2:
        print('!!! Environment/configuration error; see the log above.')

In [ ]:
# === 7. Remove checkpoints, show progress, package ===
if os.path.isdir(RESULTS):
    for res in glob.glob(os.path.join(RESULTS, '**', 'result.json'), recursive=True):
        for ck in Path(os.path.dirname(res)).glob('*.pt*'):
            ck.unlink()
    subprocess.call([sys.executable, 'run_exp.py', '--status', '--cells', *CELLS, '--out', RESULTS])
    print('Created', shutil.make_archive(f'{WORK}/results_exp_DE', 'zip', WORK, 'results_exp'))
shutil.rmtree(CODE, ignore_errors=True)
shutil.rmtree(DATA, ignore_errors=True)